# Sección nueva

Paso 1: Cargar la base de datos

In [ ]:
import pandas as pd

# Cargar dataset
df = pd.read_csv("cleaned_dataset.csv")

# Ver primeras filas
print(df.head())

Paso 2: Crear subconjuntos (20 train / 20 test)

In [ ]:
# Mezclar datos
df_sample = df.sample(n=40, random_state=42)

# Separar
train_df = df_sample.iloc[:20]
test_df = df_sample.iloc[20:]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Paso 3: Distancia Euclidiana

In [ ]:
import numpy as np

def distancia_euclidiana(v1, v2):
    return np.sqrt(np.sum((v1 - v2)**2))

# Ejemplo dado
x1 = np.array([1,106,70,28,135,34.2,0.142,22])
x2 = np.array([2,102,86,36,120,45.5,0.127,23])

dist = distancia_euclidiana(x1, x2)
print("Distancia euclidiana:", dist)

Paso 4: KNN manual (k=3)

In [ ]:
def knn_manual(train_X, train_y, test_X, k=3):
    predictions = []

    for test_point in test_X:
        distances = []

        for i in range(len(train_X)):
            dist = distancia_euclidiana(test_point, train_X[i])
            distances.append((dist, train_y[i]))

        # Ordenar por distancia
        distances.sort(key=lambda x: x[0])

        # Tomar k vecinos
        neighbors = distances[:k]

        # Votación
        votes = [n[1] for n in neighbors]
        pred = max(set(votes), key=votes.count)

        predictions.append(pred)

    return predictions


# Preparar datos
X_train = train_df.drop("Outcome", axis=1).values
y_train = train_df["Outcome"].values
X_test = test_df.drop("Outcome", axis=1).values
y_test = test_df["Outcome"].values

# Predicción
preds = knn_manual(X_train, y_train, X_test, k=3)

# Tabla resultados
for i in range(len(preds)):
    print(f"Real: {y_test[i]} | Predicho: {preds[i]}")

Paso 5: División 80/20

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

Paso 6: KNN sin escalar

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn.fit(X_train, y_train)

y_pred_raw = knn.predict(X_test)
acc_raw = accuracy_score(y_test, y_pred_raw)

print("Accuracy sin escalar:", acc_raw)

Paso 7: Normalización (Min-Max)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

knn.fit(X_train_norm, y_train)
y_pred_norm = knn.predict(X_test_norm)

acc_norm = accuracy_score(y_test, y_pred_norm)

print("Accuracy normalizado:", acc_norm)

Paso 9: Estandarización (Z-score)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

knn.fit(X_train_std, y_train)
y_pred_std = knn.predict(X_test_std)

acc_std = accuracy_score(y_test, y_pred_std)

print("Accuracy estandarizado:", acc_std)

Paso 10/11: Tabla comparativa

In [ ]:
resultados = pd.DataFrame({
    "Metodo": ["Sin escalar", "Normalizado", "Estandarizado"],
    "Accuracy": [acc_raw, acc_norm, acc_std]
})

print(resultados)

**Preguntas de reflexión**

**1. ¿Por qué es importante normalizar o estandarizar los datos antes de usar KNN?**

Es importante porque KNN se basa en distancias. Si una variable (como glucosa) tiene valores mucho más grandes que otra (como edad), dominará el cálculo y el modelo será sesgado. La normalización o estandarización equilibran la influencia de todas las variables.

**2. ¿Qué diferencias observaste en el accuracy?**


Datos crudos → menor accuracy

Normalizados/Estandarizados → mejor rendimiento

Esto ocurre porque el modelo puede medir distancias de forma más justa.

**3. ¿Qué pasa si aumentamos k?**

k pequeño → modelo más sensible al ruido (overfitting)

k grande → modelo más general (puede perder precisión)

Hay un equilibrio ideal.

**4. ¿Ventaja de implementar KNN manualmente?**

Permite entender:

Cómo se calculan las distancias

Cómo funciona la votación

La lógica interna del algoritmo

Es clave para aprendizaje.

**5. ¿Limitaciones de KNN?**

Lento con muchos datos (calcula todas las distancias)

Problemas con muchas dimensiones (curse of dimensionality)

Sensible a escalas de datos

Requiere almacenamiento de todo el dataset